# Fine-tuning YOLO11s-cls no domínio REAL (fotos do celular)

**Notebook self-contained** — rode a célula **FT-0 (bootstrap)** primeiro e ele se vira
sozinho: carrega o modelo Roboflow salvo no Drive (`soja_yolo11s_best.pt`, gerado pela
Célula 8 do `train_yolo.ipynb`), redefine `crop_single_grain`, `CLASS_NAMES`, etc.

➡️ **Depois de um restart do Colab:** rode FT-0 → FT-1 → ... → FT-5. Não precisa
re-treinar as 100 épocas do `train_yolo.ipynb` (desde que a Célula 8 tenha salvado o
modelo base no Drive).

### Aviso honesto
São ~57 fotos (15/11/19/8/4 por classe). Isso é **pouco** — espere overfitting.
Este notebook é um **probe** pra ver a tendência, não a versão de produção.
O domain shift é de aparência (fundo cinza/luz), que vive no extrator de features —
por isso treinamos um pouco além da cabeça, mas sem descongelar tudo (senão decora as 57).

In [ ]:
# FT-0 — BOOTSTRAP (rode SEMPRE que abrir num runtime novo / depois de restart)
#
# Torna este notebook SELF-CONTAINED: não depende mais da memória do train_yolo.ipynb.
# Carrega o modelo Roboflow salvo no Drive (soja_yolo11s_best.pt, gerado pela Célula 8
# do train_yolo.ipynb) — assim NÃO precisa re-treinar as 100 épocas após um restart.
#
# Se o runtime NÃO reiniciou e você veio direto do train_yolo.ipynb, pode pular esta
# célula (as variáveis já estão em memória) — mas rodar de novo não faz mal.
!pip -q install ultralytics

import os, shutil, random, glob, tempfile, pathlib, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from getpass import getpass
from ultralytics import YOLO
from huggingface_hub import HfApi, hf_hub_download

from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive'

CLASS_NAMES = [
    'Broken soybeans', 'Immature soybeans', 'Intact soybeans',
    'Skin-damaged soybeans', 'Spotted soybeans',
]
SHORT = {
    'Broken soybeans':       'broken',
    'Immature soybeans':     'immature',
    'Intact soybeans':       'intact',
    'Skin-damaged soybeans': 'skin-damaged',
    'Spotted soybeans':      'spotted',
}
short_names = [SHORT[c] for c in CLASS_NAMES]
idx_of = {name: i for i, name in enumerate(CLASS_NAMES)}

def crop_single_grain(arr):
    """Mesmo recorte do app.py / Célula 7b: maior contorno via Otsu -> bounding box."""
    h, w = arr.shape[:2]
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (7, 7), 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN,  kernel, iterations=1)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return arr
    largest = max(contours, key=cv2.contourArea)
    ratio = cv2.contourArea(largest) / (h * w)
    if ratio <= 0.02 or ratio >= 0.95:
        return arr
    x, y, bw, bh = cv2.boundingRect(largest)
    pad = max(15, int(min(bw, bh) * 0.12))
    return arr[max(0, y-pad):min(h, y+bh+pad), max(0, x-pad):min(w, x+bw+pad)]

# --- Carrega o modelo base do Roboflow salvo no Drive ---
BASE_PT = f'{SAVE_DIR}/soja_yolo11s_best.pt'
assert os.path.exists(BASE_PT), (
    f'\n❌ NÃO achei {BASE_PT}.\n'
    '   Isso é o modelo base do Roboflow, salvo pela CÉLULA 8 do train_yolo.ipynb.\n'
    '   Se você não rodou a Célula 8 antes do restart, ele se perdeu — rode o\n'
    '   train_yolo.ipynb de novo (células 1, 2, 3, 4 e 8) pra regenerá-lo no Drive.'
)
model = YOLO(BASE_PT)                       # modelo Roboflow = ponto de partida do FT
assert [model.names[i] for i in range(len(CLASS_NAMES))] == CLASS_NAMES, \
    'Ordem das classes do modelo carregado diverge de CLASS_NAMES!'
print('✅ Modelo Roboflow carregado de', BASE_PT)

# --- HF para baixar as fotos reais ---
HF_TOKEN  = getpass('Cole seu HF token (read): ')
CORR_REPO = 'Guguinhaxd/soja-correction'
api = HfApi(token=HF_TOKEN)
print('✅ Bootstrap pronto — pode rodar FT-1 em diante.')

In [ ]:
# FT-1 — Montar dataset de fine-tuning com as fotos REAIS recortadas (train/val)
import os, shutil, random, tempfile, pathlib
from huggingface_hub import hf_hub_download

REAL_DST = '/content/soja_real'
if os.path.exists(REAL_DST):
    shutil.rmtree(REAL_DST)
rng = random.Random(42)
IMG_EXTS_SET = {'.jpg', '.jpeg', '.png'}
tmpd = tempfile.mkdtemp(prefix='ft_')

all_files = list(api.list_repo_files(CORR_REPO, repo_type='dataset'))
summary = {}
for cls in CLASS_NAMES:
    fs = sorted(f for f in all_files
                if f.startswith(f'{cls}/') and pathlib.Path(f).suffix.lower() in IMG_EXTS_SET)
    rng.shuffle(fs)
    # split estratificado 80/20 (garante >=1 no val se a classe tiver >=2 fotos)
    n_val = max(1, round(len(fs) * 0.2)) if len(fs) >= 2 else 0
    val_set = set(fs[:n_val])
    for f in fs:
        local = hf_hub_download(CORR_REPO, f, repo_type='dataset', token=HF_TOKEN, local_dir=tmpd)
        arr  = np.array(Image.open(local).convert('RGB'))
        crop = crop_single_grain(arr)
        split   = 'val' if f in val_set else 'train'
        outdir  = os.path.join(REAL_DST, split, cls)
        os.makedirs(outdir, exist_ok=True)
        Image.fromarray(crop).save(os.path.join(outdir, pathlib.Path(f).stem + '.jpg'), quality=95)
    summary[SHORT[cls]] = (len(fs) - n_val, n_val)

print('classe -> (train, val)')
for k, v in summary.items():
    print(f'  {k:>13}: {v}')
print('\n⚠️  spotted/skin-damaged têm pouquíssimas fotos — o val por classe será ruidoso.')
print('   Leia o resultado como TENDÊNCIA, não como métrica precisa.')

In [ ]:
# FT-2 — Baseline: o modelo do Roboflow (ANTES do fine-tuning) no val real
import glob

def eval_on_split(yolo_model, split):
    paths, true = [], []
    for cls in CLASS_NAMES:
        for p in sorted(glob.glob(os.path.join(REAL_DST, split, cls, '*.jpg'))):
            paths.append(p); true.append(idx_of[cls])
    if not paths:
        return None, None, None
    imgs = [Image.open(p).convert('RGB') for p in paths]
    pred = [idx_of[yolo_model.names[int(r.probs.top1)]]
            for r in yolo_model.predict(imgs, imgsz=224, verbose=False)]
    true, pred = np.array(true), np.array(pred)
    return (true == pred).mean(), true, pred

base_val_acc, _, _ = eval_on_split(model, 'val')
base_tr_acc,  _, _ = eval_on_split(model, 'train')
print(f'BASELINE (Roboflow, sem fine-tuning) no domínio real:')
print(f'  train real: {base_tr_acc:.1%}')
print(f'  val real:   {base_val_acc:.1%}')

In [ ]:
# FT-3 — Fine-tuning: parte do modelo Roboflow, congela o backbone, augmentation forte
#
# FREEZE controla quanto descongela:
#   freeze=10 -> treina SÓ a cabeça (Classify). Mais seguro, mexe pouco no domain shift.
#   freeze=9  -> treina o último bloco + cabeça. Adapta mais features (recomendado p/ fundo/luz).
#   freeze=8  -> mais capacidade, mais risco de decorar as 57 fotos.
#
# Augmentation forte é a alavanca principal contra pouca foto: cria variação a partir
# do que existe e força robustez ao fundo cinza / iluminação (o domain shift real).
FREEZE = 9

# ponto de partida = pesos do treino Roboflow.
# Se veio direto do train_yolo.ipynb, usa model.trainer.best; se o runtime
# reiniciou e você rodou o FT-0, usa o BASE_PT carregado do Drive.
try:
    START_PT = str(model.trainer.best)
except (AttributeError, TypeError):
    START_PT = BASE_PT

ft = YOLO(START_PT)   # pesos do treino Roboflow como ponto de partida
ft_results = ft.train(
    data=REAL_DST,
    epochs=80,
    imgsz=224,
    batch=8,            # com ~45 fotos, batch pequeno = mais passos de gradiente
    freeze=FREEZE,
    lr0=0.0005,         # 20x menor que o treino base: nao destruir o aprendido
    patience=20,
    seed=42,
    # --- augmentation forte ---
    hsv_h=0.02, hsv_s=0.7, hsv_v=0.6,   # cor/brilho: robustez a iluminacao
    degrees=15, flipud=0.5, fliplr=0.5, # geometria: grao nao tem orientacao fixa
    translate=0.1, scale=0.5, erasing=0.4,
    project='/content/runs_soja',
    name='yolo11s_finetune',
    exist_ok=True,
    verbose=True,
)
print('\nFine-tuning concluido. save_dir:', ft.trainer.save_dir)

In [ ]:
# FT-4 — Antes x Depois no domínio real + matriz de confusão
from sklearn.metrics import classification_report, confusion_matrix

new_tr_acc,  _, _              = eval_on_split(ft, 'train')
new_val_acc, val_true, val_pred = eval_on_split(ft, 'val')

print('=== Domínio real: ANTES (Roboflow) x DEPOIS (fine-tuned) ===')
print(f'  train: {base_tr_acc:.1%}  ->  {new_tr_acc:.1%}')
print(f'  val:   {base_val_acc:.1%}  ->  {new_val_acc:.1%}')
gap = new_tr_acc - new_val_acc
print(f'\n  gap train-val depois: {gap:.1%}  ', end='')
print('(gap grande = overfitting, esperado com 57 fotos)' if gap > 0.2 else '(gap saudável)')

print('\n=== Relatório no val real (fine-tuned) ===')
print(classification_report(val_true, val_pred, target_names=short_names, zero_division=0))

cm = confusion_matrix(val_true, val_pred, labels=list(range(len(CLASS_NAMES))))
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=short_names, yticklabels=short_names,
            cmap='Greens', linewidths=0.5)
plt.xlabel('Predito'); plt.ylabel('Real')
plt.title(f'Fine-tuned no domínio real — val {new_val_acc:.0%}')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
# FT-5 (OPCIONAL) — Salvar o modelo fine-tuned no Drive (só se o val real melhorou)
from pathlib import Path
best_ft = Path(ft.trainer.best)
if new_val_acc is not None and new_val_acc > (base_val_acc or 0):
    dest = f'{SAVE_DIR}/soja_yolo11s_finetuned.pt'
    shutil.copy(best_ft, dest)
    print(f'✅ val real melhorou ({base_val_acc:.1%} -> {new_val_acc:.1%}) — salvo em {dest}')
else:
    print(f'❌ val real NÃO melhorou ({base_val_acc:.1%} -> {new_val_acc:.1%}) — não salvei.')
    print('   Provável: 57 fotos é pouco pra esse domain shift. Colete mais ou padronize a captura.')